# Part 2: Add grounding from web results

In Part 1, you built a knowledge base with indexed data. Now you'll connect a knowledge base to the new MAI grounding service which provides results from the web. That way, your knowledge base can provide answers for questions that require up-to-date news or topics outside the internal data.

## Step 1: Set up environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has *almost* everything you need: Azure AI Search endpoints, Azure OpenAI credentials, and model configuration.

‼️ You also need an `MAI_GROUNDING_KEY` variable. Your lab proctors will provide a link to a file that has that variable. Add it to the bottom of your current `.env`.

```shell
MAI_GROUNDING_KEY="some-value-here"
```

Then run the cell below to load in those variables:

In [17]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
MAI_GROUNDING_KEY = os.environ["MAI_GROUNDING_KEY"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


Environment variables loaded


## Step 2: Create three knowledge sources

For this knowledge base, you'll first create three knowledge sources - the same two index-based sources as you made in the first notebook, plus an additional knowledge source for the MAI grounding MCP server:

* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `web-knowledge-source`: Points to the remote MCP server for the MAI grounding service, using key-based authentication and `web` tool.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    McpServerAutoOutputParsing,
    McpServerKnowledgeSource,
    McpServerKnowledgeSourceParameters,
    McpServerStoredHeadersAuthentication,
    McpServerStoredHeadersParameters,
    McpServerTool,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WEB_KNOWLEDGE_SOURCE_NAME = "web-knowlege-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME, WEB_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "LAB532 HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "LAB532 health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

web_knowledge_source = McpServerKnowledgeSource(
    name=WEB_KNOWLEDGE_SOURCE_NAME,
    description="LAB532 MAI Grounding MCP knowledge source",
    mcp_server_parameters=McpServerKnowledgeSourceParameters(
        server_url="https://api.microsoft.ai/v3/mcp",
        authentication=McpServerStoredHeadersAuthentication(
            stored_headers_parameters=McpServerStoredHeadersParameters(
                {"headers": {"x-apikey": MAI_GROUNDING_KEY}}
            )
        ),
        tools=[McpServerTool(name="web", output_parsing=McpServerAutoOutputParsing())],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=web_knowledge_source)
print("Knowledge sources created")


## Step 3: Create the multi-source + web knowledge base

A knowledge base is the orchestration layer that combines:

1. Your data sources (the knowledge sources from Step 2)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format response

For this notebook, we are using an `outputMode` of "answerSynthesis" so that the attached LLM can also answer the original user query. We are also using a `retrievalReasoningEffort` of "low", which means that the LLM will be used for query planning and knowledge source selection.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-web-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT.rstrip("/") + "/",
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="LAB532 KB combining restored indexes and MAI Grounding MCP",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use the restored HR and health indexes for company policy questions. Use MAI Grounding for current public web context.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## Step 4: Query the knowledge base

Since this knowledge base has the ability to search the web, ask a question that requires web results:

"Answer with citations: what employee health benefits are described in the company docs, and what is Azure AI Search knowledge base preview?"

The knowledge base uses agentic retrieval to:

1. Analyze the query to understand two different topics
2. Decompose the query into focused subqueries
3. Determine which knowledge sources are relevant for each subquery
4. Run searches concurrently against the selected sources
5. Merge the results with the semanting re-ranking model

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeSourceParams,
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
web_knowledge_source_params = KnowledgeSourceParams(
    knowledge_source_name=WEB_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    kind="mcpServer",
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(
                    text="Answer with citations: what employee health benefits are described in the company docs, "
                    "and what is Azure AI Search knowledge base preview (search the web to find out)?"
                )
            ],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params, web_knowledge_source_params],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)

# Extract synthesized answer from response messages
for message in result.response or []:
    for content in message.content or []:
        text = getattr(content, "text", None)
        if text:
            display(Markdown(text))


### Review the references

The next cell prints the raw `references` array returned by the knowledge base.

The references shows which sources were retrieved, and the source metadata for each reference. Every result contains a `rerankerScore` from the re-ranking step. The results from the index include the search index fields, and the results from MAI grounding contain the title, URL, and truncated webpage snippet.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))


## Bonus: Query from the Copilot CLI

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or GitHub Copilot CLI.
If you want to try out querying the knowledge base from the Copilot CLI, use the generated MCP configuration below and follow the steps in the [Copilot CLI sidequest](copilot-cli-sidequest.md).

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
headers = {"api-key": AZURE_SEARCH_ADMIN_KEY}
config = {"mcpServers": {KNOWLEDGE_BASE_NAME: {"type": "http", "url": mcp_url, "headers": headers}}}
print(json.dumps(config, indent=2))

➡️ Continue to: [Part 3: Add Fabric IQ](part3-fabric-iq-to-kb.ipynb).